# SSL Task Comparison Summary, ps32

This notebook compares three ps32 ResNet-18 models under the same 5-cluster spatial cross-validation protocol: scratch training, masked-reconstruction SSL pretraining, and SimCLR-style contrastive SSL pretraining. It only reads existing metrics and prediction files, then writes summary tables and publication-quality figures.


## 1. Purpose and experiment configuration

No training, fine-tuning, map inference, or checkpoint modification is performed here. Primary evaluation is based on fold-wise SCV metrics; pooled ROC/PR curves are generated only as visualization aids.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

patch_size = 32
patch_tag = "ps32"

summary_output_dir = PROJECT_ROOT / "outputs" / "R1_ssl_task_comparison" / "summary"
summary_figure_dir = PROJECT_ROOT / "figures" / "R1_ssl_task_comparison" / "summary"
combined_fold_metrics_path = summary_output_dir / "ssl_task_comparison_ps32_fold_metrics.csv"
summary_metrics_path = summary_output_dir / "ssl_task_comparison_ps32_summary_metrics.csv"
interpretation_path = summary_output_dir / "ssl_task_comparison_ps32_interpretation.csv"

model_configs = {
    "scratch_resnet18": {
        "model_display_name": "Scratch",
        "model_key": "scratch_resnet18",
        "pretraining": "none",
        "fold_metrics": PROJECT_ROOT / "outputs" / "R0_resnet18_scratch_baseline" / "metrics" / "scratch_resnet18_ps32_fold_metrics.csv",
        "summary_metrics": PROJECT_ROOT / "outputs" / "R0_resnet18_scratch_baseline" / "metrics" / "scratch_resnet18_ps32_summary_metrics.csv",
        "prediction_pattern": PROJECT_ROOT / "outputs" / "R0_resnet18_scratch_baseline" / "predictions" / "scratch_resnet18_ps32_fold{fold}_predictions.csv",
    },
    "masked_recon_resnet18": {
        "model_display_name": "Masked recon.",
        "model_key": "masked_recon_resnet18",
        "pretraining": "masked_recon",
        "fold_metrics": PROJECT_ROOT / "outputs" / "R1_ssl_task_comparison" / "masked_recon" / "metrics" / "masked_recon_resnet18_ps32_fold_metrics.csv",
        "summary_metrics": PROJECT_ROOT / "outputs" / "R1_ssl_task_comparison" / "masked_recon" / "metrics" / "masked_recon_resnet18_ps32_summary_metrics.csv",
        "prediction_pattern": PROJECT_ROOT / "outputs" / "R1_ssl_task_comparison" / "masked_recon" / "predictions" / "masked_recon_resnet18_ps32_fold{fold}_predictions.csv",
    },
    "contrastive_resnet18": {
        "model_display_name": "Contrastive",
        "model_key": "contrastive_resnet18",
        "pretraining": "contrastive",
        "fold_metrics": PROJECT_ROOT / "outputs" / "R1_ssl_task_comparison" / "contrastive" / "metrics" / "contrastive_resnet18_ps32_fold_metrics.csv",
        "summary_metrics": PROJECT_ROOT / "outputs" / "R1_ssl_task_comparison" / "contrastive" / "metrics" / "contrastive_resnet18_ps32_summary_metrics.csv",
        "prediction_pattern": PROJECT_ROOT / "outputs" / "R1_ssl_task_comparison" / "contrastive" / "predictions" / "contrastive_resnet18_ps32_fold{fold}_predictions.csv",
    },
}

palette = {
    "Scratch": "#4D4D4D",
    "Masked recon.": "#0072B2",
    "Contrastive": "#D55E00",
}
model_order = ["Scratch", "Masked recon.", "Contrastive"]
print("Project root:", PROJECT_ROOT)
print("Summary output dir:", summary_output_dir)
print("Summary figure dir:", summary_figure_dir)


Project root: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project
Summary output dir: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary
Summary figure dir: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary


## 2. Import packages and project modules


In [2]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.metrics import safe_average_precision, safe_roc_auc
from src.plotting import set_publication_plot_style
from src.utils import ensure_dir

ensure_dir(summary_output_dir)
ensure_dir(summary_figure_dir)


WindowsPath('D:/SBU first year/First year paper/LSM_SSL_ResNet18_Project/figures/R1_ssl_task_comparison/summary')

## 3. Set paths and plotting style


In [3]:
applied_font = set_publication_plot_style(font_family="Arial", font_size=10)
print("Applied plotting font:", applied_font)
print("All figures use font size 10, PNG dpi 600, and PDF TrueType font embedding.")


Applied plotting font: Arial
All figures use font size 10, PNG dpi 600, and PDF TrueType font embedding.


## 4. Load fold metrics for all models


In [4]:
required_fold_columns = [
    "model_display_name", "model_key", "pretraining", "patch_size", "fold", "n_train", "n_val", "n_test",
    "n_test_pos", "n_test_neg", "auc", "pr_auc", "accuracy_05", "precision_05", "recall_05", "f1_05",
    "best_threshold_f1", "accuracy_best_f1", "precision_best_f1", "recall_best_f1", "f1_best_f1",
    "tn", "fp", "fn", "tp", "best_epoch", "best_val_auc", "best_val_loss",
]

fold_metric_parts = []
for key, cfg in model_configs.items():
    if not cfg["fold_metrics"].exists():
        raise FileNotFoundError(cfg["fold_metrics"])
    metrics = pd.read_csv(cfg["fold_metrics"])
    metrics["model_display_name"] = cfg["model_display_name"]
    metrics["model_key"] = cfg["model_key"]
    metrics["pretraining"] = cfg["pretraining"]
    if "patch_size" not in metrics.columns:
        metrics["patch_size"] = patch_size
    metrics["patch_size"] = metrics["patch_size"].astype(int)
    missing = [column for column in required_fold_columns if column not in metrics.columns]
    if missing:
        raise ValueError(f"{key} fold metrics missing columns: {missing}")
    fold_metric_parts.append(metrics[required_fold_columns].copy())

combined_fold_metrics = pd.concat(fold_metric_parts, ignore_index=True)
combined_fold_metrics["model_display_name"] = pd.Categorical(combined_fold_metrics["model_display_name"], categories=model_order, ordered=True)
combined_fold_metrics = combined_fold_metrics.sort_values(["model_display_name", "fold"]).reset_index(drop=True)
print(combined_fold_metrics[["model_display_name", "fold", "auc", "pr_auc", "f1_05"]].to_string(index=False))


model_display_name  fold      auc   pr_auc    f1_05
           Scratch     0 0.624023 0.635310 0.562500
           Scratch     1 0.769231 0.788888 0.509091
           Scratch     2 0.754361 0.677231 0.742424
           Scratch     3 0.846939 0.840001 0.812500
           Scratch     4 0.899654 0.911957 0.692308
     Masked recon.     0 0.895508 0.877846 0.756098
     Masked recon.     1 0.786325 0.795165 0.526316
     Masked recon.     2 0.799217 0.710585 0.731034
     Masked recon.     3 0.811224 0.850065 0.785714
     Masked recon.     4 0.812284 0.835577 0.754098
       Contrastive     0 0.872070 0.819683 0.818182
       Contrastive     1 0.704142 0.714570 0.646154
       Contrastive     2 0.910644 0.883535 0.842975
       Contrastive     3 0.806122 0.848101 0.750000
       Contrastive     4 0.897059 0.889842 0.571429


## 5. Load fold prediction files for all models


In [5]:
required_prediction_columns = [
    "sample_id", "x", "y", "label", "source", "cluster_id", "fold", "y_true", "y_logit", "y_prob",
    "y_pred_05", "y_pred_best_f1", "split",
]

prediction_parts = []
predictions_by_model_fold = {}
for key, cfg in model_configs.items():
    for fold in range(5):
        path = Path(str(cfg["prediction_pattern"]).format(fold=fold))
        if not path.exists():
            raise FileNotFoundError(path)
        pred = pd.read_csv(path)
        missing = [column for column in required_prediction_columns if column not in pred.columns]
        if missing:
            raise ValueError(f"{path} missing columns: {missing}")
        pred = pred[required_prediction_columns].copy()
        pred["model_display_name"] = cfg["model_display_name"]
        pred["model_key"] = cfg["model_key"]
        pred["pretraining"] = cfg["pretraining"]
        prediction_parts.append(pred)
        predictions_by_model_fold[(cfg["model_display_name"], fold)] = pred

all_predictions = pd.concat(prediction_parts, ignore_index=True)
print("Loaded prediction rows:", len(all_predictions))
print(all_predictions.groupby(["model_display_name", "fold"]).size().to_string())


Loaded prediction rows: 1032
model_display_name  fold
Contrastive         0        64
                    1        78
                    2       106
                    3        28
                    4        68
Masked recon.       0        64
                    1        78
                    2       106
                    3        28
                    4        68
Scratch             0        64
                    1        78
                    2       106
                    3        28
                    4        68


## 6. Validate input metrics and predictions


In [6]:
warnings = []
for display_name in model_order:
    model_metrics = combined_fold_metrics.loc[combined_fold_metrics["model_display_name"] == display_name]
    folds = sorted(model_metrics["fold"].astype(int).unique().tolist())
    if folds != [0, 1, 2, 3, 4]:
        raise ValueError(f"{display_name} does not have folds 0..4: {folds}")
    if len(model_metrics) != 5:
        raise ValueError(f"{display_name} should have exactly 5 fold metric rows.")
    model_preds = all_predictions.loc[all_predictions["model_display_name"] == display_name]
    pred_folds = sorted(model_preds["fold"].astype(int).unique().tolist())
    if pred_folds != [0, 1, 2, 3, 4]:
        raise ValueError(f"{display_name} does not have prediction folds 0..4: {pred_folds}")
    if not model_preds["y_prob"].between(0, 1).all():
        raise ValueError(f"{display_name} has y_prob outside [0, 1].")
    if set(model_preds["y_true"].astype(int).unique().tolist()) - {0, 1}:
        raise ValueError(f"{display_name} has non-binary y_true values.")
    for fold in range(5):
        pred = predictions_by_model_fold[(display_name, fold)]
        if pred["sample_id"].duplicated().any():
            raise ValueError(f"{display_name} fold {fold} has duplicate sample_id values.")

for fold in range(5):
    reference_ids = None
    reference_counts = None
    for display_name in model_order:
        pred = predictions_by_model_fold[(display_name, fold)]
        ids = set(pred["sample_id"].astype(str))
        counts = (int((pred["y_true"] == 1).sum()), int((pred["y_true"] == 0).sum()))
        if reference_ids is None:
            reference_ids = ids
            reference_counts = counts
        else:
            if ids != reference_ids:
                warnings.append(f"Fold {fold}: test sample IDs differ for {display_name}.")
            if counts != reference_counts:
                warnings.append(f"Fold {fold}: n_test_pos/n_test_neg differ for {display_name}.")

if warnings:
    print("Validation warnings:")
    for warning in warnings:
        print("WARNING:", warning)
else:
    print("Validation passed: folds, predictions, probabilities, labels, sample IDs, and test counts are consistent across models.")


Validation passed: folds, predictions, probabilities, labels, sample IDs, and test counts are consistent across models.


## 7. Create combined fold metrics table


In [7]:
combined_fold_metrics.to_csv(combined_fold_metrics_path, index=False)
print("Saved combined fold metrics:", combined_fold_metrics_path)
print(combined_fold_metrics_path.exists())


Saved combined fold metrics: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary\ssl_task_comparison_ps32_fold_metrics.csv
True


## 8. Create summary metrics table


In [8]:
summary_rows = []
for display_name in model_order:
    part = combined_fold_metrics.loc[combined_fold_metrics["model_display_name"] == display_name].copy()
    row = {
        "model_display_name": display_name,
        "model_key": part["model_key"].iloc[0],
        "pretraining": part["pretraining"].iloc[0],
        "patch_size": int(part["patch_size"].iloc[0]),
        "mean_auc": float(part["auc"].mean()),
        "std_auc": float(part["auc"].std(ddof=1)),
        "median_auc": float(part["auc"].median()),
        "min_auc": float(part["auc"].min()),
        "max_auc": float(part["auc"].max()),
        "worst_fold_auc": float(part["auc"].min()),
        "best_fold_auc": float(part["auc"].max()),
        "mean_pr_auc": float(part["pr_auc"].mean()),
        "std_pr_auc": float(part["pr_auc"].std(ddof=1)),
        "mean_f1_05": float(part["f1_05"].mean()),
        "std_f1_05": float(part["f1_05"].std(ddof=1)),
        "mean_recall_05": float(part["recall_05"].mean()),
        "std_recall_05": float(part["recall_05"].std(ddof=1)),
        "mean_precision_05": float(part["precision_05"].mean()),
        "std_precision_05": float(part["precision_05"].std(ddof=1)),
        "mean_accuracy_05": float(part["accuracy_05"].mean()),
        "std_accuracy_05": float(part["accuracy_05"].std(ddof=1)),
        "mean_f1_best": float(part["f1_best_f1"].mean()),
        "std_f1_best": float(part["f1_best_f1"].std(ddof=1)),
    }
    summary_rows.append(row)

summary_metrics = pd.DataFrame(summary_rows)
scratch_row = summary_metrics.loc[summary_metrics["model_display_name"] == "Scratch"].iloc[0]
summary_metrics["delta_mean_auc_vs_scratch"] = summary_metrics["mean_auc"] - float(scratch_row["mean_auc"])
summary_metrics["delta_std_auc_vs_scratch"] = summary_metrics["std_auc"] - float(scratch_row["std_auc"])
summary_metrics["delta_worst_auc_vs_scratch"] = summary_metrics["worst_fold_auc"] - float(scratch_row["worst_fold_auc"])
summary_metrics["rank_by_mean_auc"] = summary_metrics["mean_auc"].rank(ascending=False, method="min").astype(int)
summary_metrics["rank_by_stability"] = summary_metrics["std_auc"].rank(ascending=True, method="min").astype(int)
summary_metrics["rank_by_worst_fold_auc"] = summary_metrics["worst_fold_auc"].rank(ascending=False, method="min").astype(int)

required_summary_columns = [
    "model_display_name", "model_key", "pretraining", "patch_size", "mean_auc", "std_auc", "median_auc",
    "min_auc", "max_auc", "worst_fold_auc", "best_fold_auc", "mean_pr_auc", "std_pr_auc", "mean_f1_05",
    "std_f1_05", "mean_recall_05", "std_recall_05", "mean_precision_05", "std_precision_05",
    "mean_accuracy_05", "std_accuracy_05", "mean_f1_best", "std_f1_best", "delta_mean_auc_vs_scratch",
    "delta_std_auc_vs_scratch", "delta_worst_auc_vs_scratch", "rank_by_mean_auc", "rank_by_stability",
    "rank_by_worst_fold_auc",
]
summary_metrics = summary_metrics[required_summary_columns]
summary_metrics.to_csv(summary_metrics_path, index=False)
print("Saved summary metrics:", summary_metrics_path)
print(summary_metrics[["model_display_name", "mean_auc", "std_auc", "worst_fold_auc", "rank_by_mean_auc", "rank_by_stability"]].to_string(index=False))


Saved summary metrics: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary\ssl_task_comparison_ps32_summary_metrics.csv
model_display_name  mean_auc  std_auc  worst_fold_auc  rank_by_mean_auc  rank_by_stability
           Scratch  0.778842 0.104754        0.624023                 3                  3
     Masked recon.  0.820912 0.043011        0.786325                 2                  1
       Contrastive  0.838008 0.084941        0.704142                 1                  2


## 9. Create interpretation table


In [9]:
interpretation = pd.DataFrame(
    [
        {
            "model_display_name": "Scratch",
            "main_strength": "Reference supervised baseline trained from random initialization.",
            "main_limitation": "Spatial fold variability is high, with the weakest worst-fold AUC.",
            "recommended_interpretation": "Use as the baseline for quantifying SSL gains under spatial cross-validation.",
        },
        {
            "model_display_name": "Masked recon.",
            "main_strength": "Strongest spatial generalization stability and best worst-fold AUC.",
            "main_limitation": "Mean AUC is slightly lower than contrastive learning in this run.",
            "recommended_interpretation": "Best evidence for robust spatial generalization across heterogeneous folds.",
        },
        {
            "model_display_name": "Contrastive",
            "main_strength": "Highest mean AUC across SCV folds.",
            "main_limitation": "Less stable than masked reconstruction and lower worst-fold AUC than masked reconstruction.",
            "recommended_interpretation": "Best average discrimination, but with more fold-to-fold variability than masked reconstruction.",
        },
    ]
)
interpretation.to_csv(interpretation_path, index=False)
print("Saved interpretation table:", interpretation_path)
print(interpretation.to_string(index=False))


Saved interpretation table: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary\ssl_task_comparison_ps32_interpretation.csv
model_display_name                                                       main_strength                                                                             main_limitation                                                                      recommended_interpretation
           Scratch   Reference supervised baseline trained from random initialization.                          Spatial fold variability is high, with the weakest worst-fold AUC.                   Use as the baseline for quantifying SSL gains under spatial cross-validation.
     Masked recon. Strongest spatial generalization stability and best worst-fold AUC.                           Mean AUC is slightly lower than contrastive learning in this run.                     Best evidence for robust spatial generalization across heterogeneous folds.
    

## 10. Generate publication-quality comparison figures


In [10]:
def save_png_pdf(fig, stem: str):
    png_path = summary_figure_dir / f"{stem}.png"
    pdf_path = summary_figure_dir / f"{stem}.pdf"
    fig.savefig(png_path, dpi=600, bbox_inches="tight", transparent=False, facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", transparent=False, facecolor="white")
    plt.close(fig)
    return png_path, pdf_path


def clean_axis(ax, grid_y=True):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_y:
        ax.grid(axis="y", color="#E6E6E6", linewidth=0.7)
        ax.set_axisbelow(True)


def roc_curve_points(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    thresholds = np.r_[np.inf, np.sort(np.unique(y_prob))[::-1], -np.inf]
    positives = max(int((y_true == 1).sum()), 1)
    negatives = max(int((y_true == 0).sum()), 1)
    tpr, fpr = [], []
    for threshold in thresholds:
        pred = y_prob >= threshold
        tp = int(((y_true == 1) & pred).sum())
        fp = int(((y_true == 0) & pred).sum())
        tpr.append(tp / positives)
        fpr.append(fp / negatives)
    return np.asarray(fpr), np.asarray(tpr)


def pr_curve_points(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    thresholds = np.r_[np.inf, np.sort(np.unique(y_prob))[::-1], -np.inf]
    positives = max(int((y_true == 1).sum()), 1)
    precision, recall = [], []
    for threshold in thresholds:
        pred = y_prob >= threshold
        tp = int(((y_true == 1) & pred).sum())
        fp = int(((y_true == 0) & pred).sum())
        precision.append(tp / (tp + fp) if (tp + fp) else 1.0)
        recall.append(tp / positives)
    return np.asarray(recall), np.asarray(precision)

saved_figures = []

# Fig R1a: mean AUC with fold variability.
fig, ax = plt.subplots(figsize=(5.8, 4.0))
x = np.arange(len(model_order))
means = [summary_metrics.loc[summary_metrics["model_display_name"] == name, "mean_auc"].iloc[0] for name in model_order]
stds = [summary_metrics.loc[summary_metrics["model_display_name"] == name, "std_auc"].iloc[0] for name in model_order]
colors = [palette[name] for name in model_order]
ax.bar(x, means, yerr=stds, color=colors, alpha=0.88, capsize=4, edgecolor="white", linewidth=0.8)
rng = np.random.default_rng(42)
for i, name in enumerate(model_order):
    aucs = combined_fold_metrics.loc[combined_fold_metrics["model_display_name"] == name, "auc"].to_numpy()
    jitter = rng.normal(0, 0.035, size=len(aucs))
    ax.scatter(np.full_like(aucs, x[i], dtype=float) + jitter, aucs, s=22, color="white", edgecolor=palette[name], linewidth=0.9, zorder=3)
ax.axhline(0.5, color="#777777", linestyle="--", linewidth=1.0)
scratch_mean = means[0]
for i, name in enumerate(model_order[1:], start=1):
    delta = means[i] - scratch_mean
    ax.text(x[i], means[i] + stds[i] + 0.025, f"{delta:+.3f}", ha="center", va="bottom", fontsize=10, color=palette[name])
ax.set_xticks(x)
ax.set_xticklabels(model_order)
ax.set_ylabel("AUC")
ax.set_ylim(0.45, 1.0)
ax.set_title("Mean AUC across SCV folds")
clean_axis(ax)
saved_figures.extend(save_png_pdf(fig, "Fig_R1a_mean_auc_bar_error_ps32"))

# Fig R1b: fold-wise AUC lines.
fig, ax = plt.subplots(figsize=(6.0, 4.0))
for name in model_order:
    part = combined_fold_metrics.loc[combined_fold_metrics["model_display_name"] == name].sort_values("fold")
    ax.plot(part["fold"], part["auc"], marker="o", color=palette[name], label=name)
ax.axhline(0.5, color="#777777", linestyle="--", linewidth=1.0)
ax.set_xticks([0, 1, 2, 3, 4])
ax.set_xlabel("SCV fold")
ax.set_ylabel("AUC")
ax.set_ylim(0.45, 1.0)
ax.set_title("Fold-wise AUC")
ax.legend(frameon=False)
clean_axis(ax)
saved_figures.extend(save_png_pdf(fig, "Fig_R1b_foldwise_auc_lines_ps32"))

# Fig R1c: worst-fold and stability.
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.8), constrained_layout=True)
worst = [summary_metrics.loc[summary_metrics["model_display_name"] == name, "worst_fold_auc"].iloc[0] for name in model_order]
stability = [summary_metrics.loc[summary_metrics["model_display_name"] == name, "std_auc"].iloc[0] for name in model_order]
axes[0].bar(x, worst, color=colors, alpha=0.88, edgecolor="white", linewidth=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_order, rotation=20, ha="right")
axes[0].set_ylabel("Worst-fold AUC")
axes[0].set_ylim(0.45, 1.0)
axes[0].set_title("(a) Worst-fold performance")
for i, value in enumerate(worst):
    axes[0].text(i, value + 0.015, f"{value:.3f}", ha="center", va="bottom", fontsize=10)
clean_axis(axes[0])
axes[1].bar(x, stability, color=colors, alpha=0.88, edgecolor="white", linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_order, rotation=20, ha="right")
axes[1].set_ylabel("AUC std. across folds")
axes[1].set_ylim(0.0, max(stability) * 1.35)
axes[1].set_title("(b) Fold-to-fold variability")
for i, value in enumerate(stability):
    axes[1].text(i, value + max(stability) * 0.035, f"{value:.3f}", ha="center", va="bottom", fontsize=10)
clean_axis(axes[1])
saved_figures.extend(save_png_pdf(fig, "Fig_R1c_worstfold_stability_ps32"))

print("WARNING: Pooled ROC/PR curves are used only for visualization. Primary evaluation is based on fold-wise SCV metrics.")

# Fig R1d: pooled ROC.
fig, ax = plt.subplots(figsize=(5.2, 4.2))
for name in model_order:
    pred = all_predictions.loc[all_predictions["model_display_name"] == name]
    y_true = pred["y_true"].to_numpy()
    y_prob = pred["y_prob"].to_numpy()
    fpr, tpr = roc_curve_points(y_true, y_prob)
    auc = safe_roc_auc(y_true, y_prob)
    ax.plot(fpr, tpr, color=palette[name], label=f"{name} AUC={auc:.3f}")
ax.plot([0, 1], [0, 1], color="#777777", linestyle="--", linewidth=1.0)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Pooled ROC curves")
ax.legend(frameon=False)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
clean_axis(ax)
saved_figures.extend(save_png_pdf(fig, "Fig_R1d_pooled_roc_all_models_ps32"))

# Fig R1e: pooled PR.
fig, ax = plt.subplots(figsize=(5.2, 4.2))
for name in model_order:
    pred = all_predictions.loc[all_predictions["model_display_name"] == name]
    y_true = pred["y_true"].to_numpy()
    y_prob = pred["y_prob"].to_numpy()
    recall, precision = pr_curve_points(y_true, y_prob)
    ap = safe_average_precision(y_true, y_prob)
    ax.plot(recall, precision, color=palette[name], label=f"{name} AP={ap:.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Pooled precision-recall curves")
ax.legend(frameon=False)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
clean_axis(ax)
saved_figures.extend(save_png_pdf(fig, "Fig_R1e_pooled_pr_all_models_ps32"))

# Fig R1f: compact heatmap.
heatmap_columns = ["mean_auc", "std_auc", "worst_fold_auc", "mean_pr_auc", "mean_f1_05"]
heatmap_labels = ["mean AUC", "std AUC", "worst AUC", "mean PR-AUC", "mean F1@0.5"]
heat_raw = summary_metrics.set_index("model_display_name").loc[model_order, heatmap_columns]
heat_display = heat_raw.copy()
for column in heatmap_columns:
    values = heat_raw[column].to_numpy(dtype=float)
    if np.isclose(values.max(), values.min()):
        scaled = np.full_like(values, 0.5)
    else:
        scaled = (values - values.min()) / (values.max() - values.min())
    if column == "std_auc":
        scaled = 1.0 - scaled
    heat_display[column] = scaled
fig, ax = plt.subplots(figsize=(7.0, 2.8))
im = ax.imshow(heat_display.to_numpy(dtype=float), cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(np.arange(len(heatmap_columns)))
ax.set_xticklabels(heatmap_labels)
ax.set_yticks(np.arange(len(model_order)))
ax.set_yticklabels(model_order)
ax.set_title("Compact metric summary")
for i, name in enumerate(model_order):
    for j, column in enumerate(heatmap_columns):
        value = heat_raw.loc[name, column]
        ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=10, color="#1A1A1A")
for spine in ax.spines.values():
    spine.set_linewidth(0.8)
fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025).ax.tick_params(labelsize=10)
fig.tight_layout()
saved_figures.extend(save_png_pdf(fig, "Fig_R1f_metric_heatmap_ps32"))

print("Saved figures:")
for path in saved_figures:
    print(path)


Saved figures:
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1a_mean_auc_bar_error_ps32.png
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1a_mean_auc_bar_error_ps32.pdf
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1b_foldwise_auc_lines_ps32.png
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1b_foldwise_auc_lines_ps32.pdf
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1c_worstfold_stability_ps32.png
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1c_worstfold_stability_ps32.pdf
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1d_pooled_roc_all_models_ps32.png
D:\SBU first year\First year paper\

## 11. Print key scientific findings


In [11]:
scratch = summary_metrics.loc[summary_metrics["model_display_name"] == "Scratch"].iloc[0]
masked = summary_metrics.loc[summary_metrics["model_display_name"] == "Masked recon."].iloc[0]
contrastive = summary_metrics.loc[summary_metrics["model_display_name"] == "Contrastive"].iloc[0]

print("Key scientific findings")
print(f"Scratch: mean AUC = {scratch['mean_auc']:.3f}, std AUC = {scratch['std_auc']:.3f}, worst-fold AUC = {scratch['worst_fold_auc']:.3f}")
print(f"Masked recon.: mean AUC = {masked['mean_auc']:.3f}, std AUC = {masked['std_auc']:.3f}, worst-fold AUC = {masked['worst_fold_auc']:.3f}")
print(f"Contrastive: mean AUC = {contrastive['mean_auc']:.3f}, std AUC = {contrastive['std_auc']:.3f}, worst-fold AUC = {contrastive['worst_fold_auc']:.3f}")
print("Both SSL methods improve mean AUC over scratch.")
print("Contrastive achieves the highest mean AUC.")
print("Masked reconstruction achieves the lowest fold-to-fold variability and the highest worst-fold AUC.")
print("Therefore, contrastive pretraining improves average discrimination, whereas masked reconstruction provides stronger spatial generalization stability.")


Key scientific findings
Scratch: mean AUC = 0.779, std AUC = 0.105, worst-fold AUC = 0.624
Masked recon.: mean AUC = 0.821, std AUC = 0.043, worst-fold AUC = 0.786
Contrastive: mean AUC = 0.838, std AUC = 0.085, worst-fold AUC = 0.704
Both SSL methods improve mean AUC over scratch.
Contrastive achieves the highest mean AUC.
Masked reconstruction achieves the lowest fold-to-fold variability and the highest worst-fold AUC.
Therefore, contrastive pretraining improves average discrimination, whereas masked reconstruction provides stronger spatial generalization stability.


## 12. Save all outputs and final QA summary


In [12]:
required_files = [combined_fold_metrics_path, summary_metrics_path, interpretation_path, *saved_figures]
missing_files = [path for path in required_files if not Path(path).exists()]
if missing_files:
    raise FileNotFoundError(missing_files)

expected_figure_stems = [
    "Fig_R1a_mean_auc_bar_error_ps32",
    "Fig_R1b_foldwise_auc_lines_ps32",
    "Fig_R1c_worstfold_stability_ps32",
    "Fig_R1d_pooled_roc_all_models_ps32",
    "Fig_R1e_pooled_pr_all_models_ps32",
    "Fig_R1f_metric_heatmap_ps32",
]
for stem in expected_figure_stems:
    for suffix in ["png", "pdf"]:
        path = summary_figure_dir / f"{stem}.{suffix}"
        if not path.exists():
            raise FileNotFoundError(path)

print("Saved summary CSV files:")
print(combined_fold_metrics_path)
print(summary_metrics_path)
print(interpretation_path)
print("Saved figures:")
for stem in expected_figure_stems:
    print(summary_figure_dir / f"{stem}.png")
    print(summary_figure_dir / f"{stem}.pdf")
print("Standardized plotting style confirmed:", applied_font, "font size 10, PNG dpi 600, PDF fonttype 42.")
print("No training, fine-tuning, model checkpoint modification, or susceptibility map generation was performed.")


Saved summary CSV files:
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary\ssl_task_comparison_ps32_fold_metrics.csv
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary\ssl_task_comparison_ps32_summary_metrics.csv
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\R1_ssl_task_comparison\summary\ssl_task_comparison_ps32_interpretation.csv
Saved figures:
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1a_mean_auc_bar_error_ps32.png
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1a_mean_auc_bar_error_ps32.pdf
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1b_foldwise_auc_lines_ps32.png
D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\R1_ssl_task_comparison\summary\Fig_R1b_foldwise_auc_lines_p